In [ ]:
!git clone -q https://github.com/jonsnow-org/Ttbik.git
%cd Ttbik/ai-system/colab/sham_small
!pip install -q tokenizers==0.23.2 soundfile==0.14.0 imageio-ffmpeg==0.5.1 python-telegram-bot==21.6

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["SHAM_TELEGRAM_BOT_TOKEN"] = UserSecretsClient().get_secret("SHAM_BOT_TOKEN")
print("Bot token loaded:", bool(os.environ["SHAM_TELEGRAM_BOT_TOKEN"]))

In [ ]:
!find /kaggle/input -iname "*.pt"
!find /kaggle/input -iname "tokenizer.json"

In [ ]:
from pathlib import Path
found = list(Path("/kaggle/input").rglob("*tokenizer*"))
print(found)

In [ ]:
!git -C /kaggle/working/Ttbik pull

In [ ]:
import subprocess, os, time, requests

env = os.environ.copy()
env["SHAM_SMALL_CHECKPOINT_PATH"] = "/kaggle/input/datasets/jonsnowjonsnow/nova-small-checkpoint/checkpoints/step_17400.pt"
env["SHAM_SMALL_TOKENIZER_PATH"] = "/kaggle/input/datasets/jonsnowjonsnow/nova-small-checkpoint/nova_small_tokenizer.json"

log_file = open("/kaggle/working/serve.log", "w")
server = subprocess.Popen(
    ["uvicorn", "serve:app", "--host", "127.0.0.1", "--port", "8000"],
    env=env, stdout=log_file, stderr=subprocess.STDOUT,
    start_new_session=True,  # يفصله عن هذه الخلية فلا يتأثر لو أوقفناها لاحقاً
)
print("تم إطلاق الخادم، PID:", server.pid)

ok = False
for _ in range(30):
    if server.poll() is not None:
        print("الخادم توقف مبكراً، رمز الخروج:", server.returncode)
        break
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            print("الخادم يعمل:", r.json())
            ok = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ok:
    print("--- آخر سجل للخادم ---")
    log_file.flush()
    print(open("/kaggle/working/serve.log").read()[-3000:])

In [ ]:
!pip install -q nest_asyncio
import nest_asyncio
nest_asyncio.apply()

import os, sys
os.environ["SHAM_SMALL_BACKEND_URL"] = "http://127.0.0.1:8000"
sys.path.insert(0, "web")
import telegram_bot
telegram_bot.main()